In [1094]:
import os
import fitz
from bs4 import BeautifulSoup
import requests
import fitz
from io import BytesIO
import pdfplumber
import time 


from supabase.client import Client, create_client
from langchain_groq import ChatGroq 
from sentence_transformers import SentenceTransformer

from langgraph.graph import StateGraph,START, END
from pydantic import BaseModel, Field # for defining structured output for models
import operator

from typing import List, TypedDict, Literal, Annotated, Optional
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

from langchain_core.messages import SystemMessage, HumanMessage

from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate

from supabase.client import Client, create_client

import warnings
warnings.filterwarnings('ignore')

In [1095]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## process the UOS student handbook 2025-2026 from Top Level Agent

In [1096]:
def bookPdfDataProcessor(BookPath): 
    uoshandBook = fitz.open(BookPath)

    Book_contentInner = []
    page_contentInnder = ""

    for page_num, page in enumerate(uoshandBook):
        blocks = page.get_text("blocks")

        for block in blocks:         
            tempCont = block[4].replace("\n", "")
            tempCont = tempCont.replace("\t", " ")
            tempCont = tempCont.replace("\xad", " ") 
            page_contentInnder = page_contentInnder + tempCont

            if len(page_contentInnder.split(" "))>250: 
                Book_contentInner.append(page_contentInnder)
                page_contentInnder = ""

    if len(page_contentInnder.split(" "))>50:
        Book_contentInner.append(page_contentInnder)
        page_contentInnder = ""

    return Book_contentInner


Book_content = bookPdfDataProcessor("./UOS Faculty Handbook 25-26 Design.pdf") + bookPdfDataProcessor("./Student Handbook.pdf")
    
print(min([len(x.split(" ")) for x in Book_content]), max([len(x.split(" ")) for x in Book_content]), sum([len(x.split(" ")) for x in Book_content])/len([len(x.split(" ")) for x in Book_content]))

139 715 299.55280898876407


### Now tokenize these texts and create a vector DB to store them

In [1097]:
topLevelAgent_QNA_vectorStore = FAISS.from_texts(Book_content, embeddings)
topLevelAgent_QNA_retriver = topLevelAgent_QNA_vectorStore.as_retriever(search_kwargs={"k":10})

In [1098]:
qa_chain_cs_docs = topLevelAgent_QNA_retriver.get_relevant_documents("what department are in College of computing and informatics? ")
# qa_chain_cs_docs = "\n\n".join([d.page_content for d in qa_chain_cs_docs])

qa_chain_cs_docs_modSnet = []
for sent in qa_chain_cs_docs: 
    for word in sent.page_content.split(" "): 
        if word == " " or word=="\t": 
            continue
        else: 
            qa_chain_cs_docs_modSnet.append(word)
        if len(qa_chain_cs_docs_modSnet)>2000: 
            break


qa_chain_cs_docs = " ".join(qa_chain_cs_docs_modSnet)
print(len(qa_chain_cs_docs.split(" ")))

2004


## define the state for the model

In [1099]:
class UOS_AgentState(BaseModel): 
    uid: str
    mensOrWomCampus: Literal['mens', 'womens']
    standing: Literal["Freshman", "Sophomore", "Junior", "Senior", "Graduate"]
    yearOfStudy: Literal[1,2,3,4,5]
    semester: Literal[1,2,3,4,5,6,7,8,9,10]
    college: Literal[
        "College of Science", 
        "College of Computing and Informatics"
    ]
    department: Literal[
        "BSc_Mathematics", 
        "BSc_Chemistry", 
        "BSc_Petroleum_Geosciences_and_Remote_Sensing", 
        "BSc_Biotechnology", 
        "BSc_Business_Information_Systems", 
        "BSc_IT_Multimedia", 
        "BSc_Computer_Science", 
        "BSc_Cybersecurity", 
        "BSc_Computer_Engineering",
        "BSc_Applied_Physics"
    ]
    studentAlreadyCompletedCourses : List[str] =[]
    StudentAlreadyCompletedCredits : int = 0
    student_query : str

    # agent decision states: below None is for just for redundancy
    UOS_AgentState_top_level_decision: Literal["ScienceAgent", "CCIAgent"] = None
    topLevelAgent_basicConversationOrNot: bool = None
    basic_Conversation_answer : str = None

    # CCI agent states
    UOS_AgentState_CCIAgent_decision: Literal["CCI_Conversation", "CCI_CourseScheduling_advising"] = None
    CCI_weblink_state : str = None
    CCI_weblink_Content : str = None
    CCI_Basic_QNA_RAG_Answer: str = None

    CCI_Course_Core_Study_Plan: List = []
    CCI_Course_Elective_Study_Plan: List = []
    CCI_Offered_Courses_this_semester: List = []

    CCI_to_Science_Communication: bool = False 
    CCI_to_Medical_communication: bool = False 
    CCI_to_OtherCollge_Communication: bool = False

    # Science agent States
    UOS_AgentState_ScienceAgent_decision: Literal["Science_Conversation", "Science_CourseScheduling_advising"] = None
    Science_weblink_state : str = None
    Science_weblink_Content : str = None
    Science_Basic_QNA_RAG_Answer: str = None

    Science_Course_Core_Study_Plan: List = []
    Science_Course_Elective_Study_Plan: List = []
    Science_Offered_Courses_this_semester: List = []

    Science_to_CCI_Communication: bool = False 
    Science_to_Medical_communication: bool = False 
    Science_to_OtherCollge_Communication: bool = False

    # Medical College course for student
    Medical_College_Offered_courses: List = []
    # Other all Colleges Offered Courses for Student
    Other_College_Offered_Courses: List = []


### define the structured schemas 

In [1100]:
class UOS_AgentState_top_level(BaseModel):
    basicConversationOrNot: bool = Field(
        description="True if the student is engaging in general conversation, False if the user is asking about academic advising or course scheduling or course registering."
    )
    academicAdvising: Optional[Literal["ScienceAgent", "CCIAgent"]] = Field(
        description="ScienceAgent if the user is from College of Science, CCIAgent if the user is from College of Computing and Informatics. Null if not applicable."
    )


class UOS_AgentState_CCI(BaseModel):
    conditionalStateGetter: Optional[Literal[
        "CCI_Conversation",
        "CCI_CourseScheduling_advising"
    ]] = Field(
        ...,
        description=(
            "'CCI_Conversation' if the student is engaging in general conversation and course related questions about the College of Computing and Informatics. "
            "If the student is asking about course scheduling or academic advising for this semester then return 'CCI_CourseScheduling_advising'."
        )
    )


class UOS_AgentState_Science(BaseModel):
    conditionalStateGetter: Optional[Literal[
        "Science_Conversation",
        "Science_CourseScheduling_advising"
    ]] = Field(
        ...,
        description=(
            "'Science_Conversation' if the student is engaging in general conversation and course related questions about the College of Science."
            "If the student is asking about course scheduling or academic advising for this semester then return 'Science_CourseScheduling_advising'."
        )
    )

### Create a CCI-Agent bottom 3 Nodes

In [1101]:
class UOS_CCI_website_Link_getter(BaseModel): 
    conditionalStateGetter: Optional[Literal[
        "B_Sc_Computer_Engineering",
        "B_Sc_Cybersecurity_Engineering",
        "B_Sc_Information_Technology_Multimedia",
        "B_Sc_Computer_Science",
        "B_Sc_Biomedical_Informatics",
        "B_Sc_Minor_in_AI",
        "B_Sc_Business_Information_system"
    ]] = Field(
        ...,
        description="Select which state to return. Must be one of the predefined values."
    )


### Create a Science-Agent bottom 3 Nodes

In [1102]:
class UOS_Science_website_Link_getter(BaseModel): 
    conditionalStateGetter: Optional[Literal[
        "B_Sc_Bio_Technology",
        "B_Sc_Petroleum_Geoscience_and_Remote_Sensing",
        "B_Sc_Applied_Physics",
        "B_Sc_Chemistry",
        "B_Sc_MatheMatics"
    ]] =  Field(
        ...,
        description="Select which state to return. Must be one of the predefined values."
    )



In [ ]:

groq_api_key = ""

generator_llm = ChatGroq(
    api_key = groq_api_key, 
    model_name = "llama-3.1-8b-instant",
    temperature = 0,
    max_completion_tokens = 512,
    top_p = 1.0,
    stream = False,
    stop = None,
    #include_reasoning  = False
)


supabase_key  = ""
supabase_url  = ""
supabaseClient = create_client(supabase_url, supabase_key)

In [1104]:
## make the model structured
topLevel_Parser = PydanticOutputParser(pydantic_object=UOS_AgentState_top_level)
topLevel_Agent_prompt = PromptTemplate(
    template=(
        "You are an academic advisor and also You will also engage in general basic conversation with students in University of Sharjah. "
        "The department in College of Science are: Department of Applied Biology offers B.Sc in Biotechnology | Department of Chemistry offers B.Sc in Chemistry | Department of Mathematics offers B.Sc in Mathematics | Department of Applied Physics and Astronomy offers B.Sc in Petroleum Geosciences and Remote Sensing and B.Sc in Applied Physics "
        "The department in College of Computing and Informatics are: Department of Computer Engineering  offers B.Sc in Computer Engineering and B.Sc in Cybersecurity Engineering | Department of Computer Science offers B.Sc in Computer Science, B.Sc in Information Technology Multimedia, B.SC in Biomedical Informatics | Department of Information Systems offers B.Sc in Business Information Systems and Minor in Data Analytics |"
        "Based on the student query information, determine if the student is engaging in general conversation related to 'College of Computing and informatics' or 'College of Science'. \n"
        "If the Query specifically related to specific college(i.e: College of Science , College of Computing and Informatics) and departments or spacific courses or if student is asking about academic advising, then  return 'basicConversationOrNot' as False and academicAdvising as True. \n"
        "Else if the query in not related to the scheduling course and academic advising or not related to specific college(i.e: College of Science , College of Computing and Informatics), then return 'basicConversationOrNot' as True. \n"

        "In summary,\n"
        "if the student query about academic advising and registering course for this semester or searching courses offered by the university this semester, respond with 'basicConversationOrNot' as False and specify which agent should handle the request in 'academicAdvising' (either 'ScienceAgent' or 'CCIAgent'). \n"
        "But if the query is about specifically related to College of Science or College of Computing and Informatics then also respond with 'basicConversationOrNot' as False and specify which agent should handle the request in 'academicAdvising' (either 'ScienceAgent' or 'CCIAgent'). \n"
        "If it's a general conversation, not specifically related to course scheduling or academic advising or college of Science or College of Computing and Informatics, respond with 'basicConversationOrNot' as True and give a brif. \n"
        "Student is now studying in College: {Student_college} , Departmental Course:{student_department}. \n"
        "If you don't have knowledge on the topic, set 'basicConversationOrNot' to True with a summary.\n\n"
        "Student query: {query}\n"
        "{format_instructions}"
        "\nDONT PROVIDE ANY EXTRA TEXT OUTSIDE OF FORMAT INSTRUCTION."
    ),
    input_variables=["Student_college", "student_department", "query"],
    partial_variables={"format_instructions": topLevel_Parser.get_format_instructions()}
)
topLevel_Agent_chain = topLevel_Agent_prompt | generator_llm | topLevel_Parser



In [1105]:
# CCI related Agents and Processor
CCI_Parser = PydanticOutputParser(pydantic_object=UOS_AgentState_CCI)
CCI_prompt = PromptTemplate(
    template=(
        "You are an academic advisor and also You will also engage in general basic conversation with students for 'College of Computing and Informatics' in University of Sharjah. \n"
        "Based on the student query, determine if the student is engaging in general conversation i.e:academic advising about College of Computing and Informatics or if they are asking about course scheduling and registering corurses for this semester. \n"
        "Here: course scheduling or registering course means: my agent should analyze the offered courses for this semester by University and scheduling courses or register the courses for the student and SEND THE REGISTER COURSES TO UNIVERSITY DATABASE. \n"
        "So if student ask YOU specifically schedule the courses for the student for this semester or registering courses for the student then select 'CCI_CourseScheduling_advising'. \n"

        # "I have a sql database where I have the offerd courses by the University for College of Computing and Informatics and student can register those courses this semeter. so  "
        "If the student query is about specifically 'schedule the courses for this semister' or 'registering course for this semester' for College of Computing and Informatics, then return 'CCI_CourseScheduling_advising' from the option and  give a brif. \n"
        "If it's a general conversation i.e:academic advising, not specifically related to 'schedule the courses for this semister' or 'registering courses for this semester' for College of Computing and Informatics, then return 'CCI_Conversation' from the given option and give a brif. \n"

        "The departments in CCI are:\n"
        "- Department of Computer Engineering → B.Sc in Computer Engineering, B.Sc in Cybersecurity Engineering\n"
        "- Department of Computer Science → B.Sc in Computer Science, B.Sc in Information Technology Multimedia, B.Sc in Biomedical Informatics\n"
        "- Department of Information Systems → B.Sc in Business Information Systems, Minor in Data Analytics\n\n"
        "Classify the student's query into one of the following options and give a brif or summary: \n\n"
        "{options}\n\n"
        "Student query: {query}\n\n"
        "{format_instructions}"
        "\n\n DONT PROVIDE ANY EXTRA TEXT OUTSIDE OF FORMAT INSTRUCTION."
    ),
    input_variables=["query", "options"],
    partial_variables={"format_instructions": CCI_Parser.get_format_instructions()}
)

CCI_Agent_chain = CCI_prompt | generator_llm | CCI_Parser

CCI_General_QNA_linkGetter = generator_llm.with_structured_output(UOS_CCI_website_Link_getter)


In [1106]:
# Science related Agents and Processor
Science_Parser = PydanticOutputParser(pydantic_object=UOS_AgentState_Science)
Science_prompt = PromptTemplate(
    template=(
        "You are an academic advisor and also You will also engage in general basic conversation with students for 'College of Science' in University of Sharjah. "
        "Based on the student query, determine if the student is engaging in general conversation and academic advising about College of Science or if they are asking about course scheduling and registering corurses for this semester. \n"
        "Here: course scheduling or registering course means: Analyze the offered courses for this semester by University and scheduling courses and register the courses for the student and SEND THE REGISTER COURSES TO UNIVERSITY DATABASE. \n"
        "So if student asks specifically schedule the courses for the student for this semester or registering courses for the student then select 'Science_CourseScheduling_advising'. \n"

        # "I have a sql database where I have the offerd courses by the University for College of Science and student can register those courses this semeter. so  "
        "If the student query is about specifically 'schedule the courses for this semister' or 'registering course for this semester' for College of Science, then return 'Science_CourseScheduling_advising' from the option and  give a brif. \n"
        "If it's a general conversation, not specifically related to 'schedule the courses for this semister' or 'registering course for this semester' for College of Science, then return 'Science_Conversation' from the given option and give a brif. \n"

        "The departments in College of Science are:\n"
        "- Department of Applied Biology offers B.Sc in Biotechnology \n"
        "- Department of Chemistry offers B.Sc in Chemistry \n"
        "- Department of Mathematics offers B.Sc in Mathematics \n"
        "- Department of Applied Physics and Astronomy offers B.Sc in Petroleum Geosciences and Remote Sensing and B.Sc in Applied Physics \n\n"
        
        "Classify the student's query into one of the following options and give a brif or summary: \n\n"
        "{options}\n\n"
        "Student query: {query}\n\n"
        "{format_instructions}"
        "\n\n DONT PROVIDE ANY EXTRA TEXT OUTSIDE OF FORMAT INSTRUCTION."
    ),
    input_variables=["query", "options"],
    partial_variables={"format_instructions": Science_Parser.get_format_instructions()}
)
Science_Agent_chain = Science_prompt | generator_llm | Science_Parser

Science_General_QNA_linkGetter = generator_llm.with_structured_output(UOS_Science_website_Link_getter)

### Now define the fuction that calls LLM on prompt and "Updates state"

In [1107]:
def topLevelLLM_call(state: UOS_AgentState): 
    # process with structured top level LLM
    response = topLevel_Agent_chain.invoke({
        "Student_college": state.college,
        "student_department": state.department,
        "query": state.student_query,
    })
    if response.basicConversationOrNot == True: 
        return {'topLevelAgent_basicConversationOrNot': True} # then end the conversation 

    return {'topLevelAgent_basicConversationOrNot': False , 'UOS_AgentState_top_level_decision': response.academicAdvising}

In [1108]:
def CCI_LLM_call(state: UOS_AgentState):
    CCI_options = [
        "CCI_Conversation",
        "CCI_CourseScheduling_advising"
    ]
    response = CCI_Agent_chain.invoke({
        "query": state.student_query,
        "options": CCI_options
    })

    #print("response from CCI:" , response)
    print("UOS_AgentState_CCIAgent_decision: ", response.conditionalStateGetter)
    return {'UOS_AgentState_CCIAgent_decision': response.conditionalStateGetter}

In [1109]:
def Science_LLM_call(state: UOS_AgentState):
    science_options = [
        "Science_Conversation",
        "Science_CourseScheduling_advising"
    ]

    response = Science_Agent_chain.invoke({
        "query": state.student_query,
        "options": science_options
    })
    print("UOS_AgentState_ScienceAgent_decision: ", response.conditionalStateGetter)
    return {'UOS_AgentState_ScienceAgent_decision': response.conditionalStateGetter}


### top-level agent: if the conversation is basic then relocate the Top-Level agen to get answer from vectorized Book Data

In [1110]:
def topLevel_agent_RAG(state: UOS_AgentState):
    # toplevelAgent -> 
    qa_chain_retrived_docs = topLevelAgent_QNA_retriver.get_relevant_documents(state.student_query)
    qa_chain_docs_2000 = []

    for sent in qa_chain_retrived_docs: 
        for word in sent.page_content.split(" "): 
            if word == " " or word=="\t": 
                continue
            else: 
                qa_chain_docs_2000.append(word)
            if len(qa_chain_docs_2000)>2000: 
                break

    qa_chain_retrived_docs = " ".join(qa_chain_docs_2000)
    
    # now re-structure the texts using LLM model
    messages = [
        SystemMessage(content=(
            "You are an academic advisor agent for the University of Sharjah. "
            "Do NOT include <think> tags or reasoning traces. "
            "Only give the final structured answer directly, in human-readable form."
        )),
        HumanMessage(content=(
            f"Student Question: {state.student_query}\n"
            f"Retrieved texts: {qa_chain_retrived_docs}"
        ))
    ]

    # basic_Conversation_answer
    responsed_structuredLLM_texts = generator_llm.invoke(messages)

    return {
        "basic_Conversation_answer": responsed_structuredLLM_texts.content 
    }

### Create a function that implements RAG oprtation in CCI and Science Agent

In [1111]:
def cci_science_agent_website_text_processor(websiteText, chunkSize=500): 
    # this Fn is implemented the rag part happening below CCI_agent and Science_agent to process basic_QNA operation from website
    websiteText_splitted = websiteText.split(" ")
    websiteText_arr = []
    for x in range(0, len(websiteText_splitted), chunkSize): 
        start=x 
        end = x+chunkSize
        if end >=len(websiteText_splitted): 
            end = len(websiteText_splitted)
        datass = " ".join(websiteText_splitted[start : end])
        websiteText_arr.append(datass)
    
    return websiteText_arr

### Now work for College of Computung and Informatics: Basic_qna: ==============================

In [1112]:
## Course and other link for CCI 
cci_basicQna_link = {
    "B_Sc_Computer_Engineering": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Computer-Engineering" ,
    "B_Sc_Cybersecurity_Engineering": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Cybersecurity-Engineering" ,

    "B_Sc_Information_Technology_Multimedia": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Information-Technology-Multimedia", 
    "B_Sc_Computer_Science": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Computer-Science", 
    "B_Sc_Biomedical_Informatics": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Biomedical-Informatics", 
    "B_Sc_Minor_in_AI": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Minor-in-AI",

    "B_Sc_Business_Information_system": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Information-Systems"
}
cci_basicQna_Offered_Courses = {
    "Department_of_Computer_Engineering": "Department of Computer Engineering Offers 2 courses for Bachelor in Science. 1.B.Sc in Computer_Engineering and 2.B.Sc in Cybersecurity_Engineering",
    "Department_of_Computer_Science": "Department of Computer Science offeres 4 courses for Bachelor in Science. 1.B.Sc in Information Technology Multimedia , 2.B.Sc Computer Science, 3.B.Sc Biomedical Informatics, 4.B.Sc in Minor in AI",
    "Department_of_Information_system": "Departent of Information system provides one course for B.Sc levele:  B.Sc in Business Information system"
}

## Course and other link for Science college
science_basicQna_link = {
    "B_Sc_Bio_Technology": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Biotechnology", 
    "B_Sc_Petroleum_Geoscience_and_Remote_Sensing": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Petroleum-Geosciences-and-Remote-Sensing", 
    "B_Sc_Applied_Physics": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Applied-Physics", 

    "B_Sc_Chemistry": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Chemistry", 
    "B_Sc_MatheMatics": "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Mathematics"
}
science_basicQna_Offered_Courses = {
    "Department_of_Applied_Biology": "provies one course in B.Sc : B.Sc in Bio-technology", 
    "Department_of_Applied_Physics_and_Astronomy": "Provides 2 course in B.Sc: Bachelor of Science in Petroleum Geosciences and Remote Sensing, Bachelor of Science in Applied Physics",
    "Department_of_Chemistry": "Privices 1 course for B.Sc: Bachelor of Science in Chemistry", 
    "Department_of_Mathematics": "Privides 1 course for B.Sc: Bachelor of Science in Mathematics"
}

In [1113]:
def CCI_Agent_basic_qna_Node_Fn(state: UOS_AgentState):
    messages = [
        SystemMessage(content=(
            "You are an academic advisor and also You will also engage in general basic conversation with students for 'College of Computing and Informatics' in University of Sharjah. \n"

            "You will give answer from the given context. These contexts are fetched form corresponding Website of University of Sharjah."
            "The department in College of Computing and Informatics are: Department of Computer Engineering  offers B.Sc in Computer Engineering and B.Sc in Cybersecurity Engineering | Department of Computer Science offers B.Sc in Computer Science, B.Sc in Information Technology Multimedia, B.SC in Biomedical Informatics | Department of Information Systems offers B.Sc in Business Information Systems and Minor in Data Analytics |"
            "Return one or more values from the list if the student query is asking about \n"
            f"College of Computing and Informatics provides: {cci_basicQna_Offered_Courses}"
            "For example:\n"
            "- If the student query is related to -> 'Computer Engineering' → return 'B_Sc_Computer_Engineering'.\n"
            "- else if the student query is related to -> 'Cybersecurity Engineering' → return 'B_Sc_Cybersecurity_Engineering'.\n"
            "- else if the student query is related to -> 'Information Technology Multimedia' → return 'B_Sc_Information_Technology_Multimedia'.\n"
            "- else if the student query is related to -> 'Computer Science' → return 'B_Sc_Computer_Science'.\n"
            "- else if the student query is related to -> 'Biomedical Informatics' → return 'B_Sc_Biomedical_Informatics'.\n"
            "- else if the student query is related to -> 'Minor in Artificial Intelligence' or 'Minor in AI' → return 'B_Sc_Minor_in_AI'.\n"
            "- else if the student query is related to -> 'Business Information Systems' → return 'B_Sc_Business_Information_system'.\n\n"
            "Also include a brif or a summary"
        )),
        HumanMessage(content=(
            f"The student is asking: {state.student_query}. "
            "Based on this information, determine if the student is engaging in general basic conversation about College of Computing and Informatics" 
        ))    
    ]

    CCI_basicQna_Website_Link = CCI_General_QNA_linkGetter.invoke(messages)
    _CCI_webLink = cci_basicQna_link[CCI_basicQna_Website_Link.conditionalStateGetter]

    # now fetch the content from website using the "_CCI_webLink"
    webLink_response = requests.get(_CCI_webLink, timeout=60)
    webLink_response.raise_for_status()
    soup = BeautifulSoup(webLink_response.text, "html.parser")
    CCIWEbsite_text = soup.get_text(separator=" ", strip=True)

    CCIWEbsite_text = CCIWEbsite_text.replace("  ", " ")
    CCIWEbsite_text = CCIWEbsite_text.replace("\n", " ")

    
    return {'CCI_weblink_state': _CCI_webLink , 'CCI_weblink_Content': CCIWEbsite_text}

In [1114]:
def CCI_Agent_basic_qna_RAG_Processor(state: UOS_AgentState): # it will be added below CCI agent to process basic QNA using fetched text from website
    #time.sleep(40) # sleep 40 Secound
    chunked_webText = cci_science_agent_website_text_processor(state.CCI_weblink_Content, chunkSize=500)
    
    CCI_Basic_QNA_vectorStore = FAISS.from_texts(chunked_webText, embeddings)
    CCI_Basic_QNA_retriver = CCI_Basic_QNA_vectorStore.as_retriever(search_kwargs={"k":5})

    CCI_chain_docs = CCI_Basic_QNA_retriver.get_relevant_documents(state.student_query)
    CCI_chain_docs = [doc.page_content for doc in CCI_chain_docs]  # list of strings
    CCI_chain_docs = " ".join(CCI_chain_docs)

    # now re-structure the texts 
    messages = [
        SystemMessage(content=(
            "You are an academic advisor agent for College of Computing and Informatics in the University of Sharjah."
            "You Use RAG system to generate answer for student and You give answer for student query by suing the given fetched text from university website. "
            "Do NOT include <think> tags or reasoning traces. "
            "Only give the final structured answer directly, in human-readable form."
        )),
        HumanMessage(content=(
            f"Student Question: {state.student_query}\n"
            f"Retrieved texts: {CCI_chain_docs}"
        ))
    ]

    # basic_Conversation_answer
    CCI_structuredLLM_texts = generator_llm.invoke(messages)

    return {
        "CCI_Basic_QNA_RAG_Answer": CCI_structuredLLM_texts.content 
    }


### Now work for college of Science: Basic_qna: ======================================

In [1115]:
def Science_Agent_basic_qna_Node_Fn(state: UOS_AgentState):
    messages = [
        SystemMessage(content=(
            "You are an a professional QNA provider for 'College of Science' in University of Sharjah. "
            "You will engage in general basic conversation with students. "
            "Always respond in a friendly and professional manner."
            "You will give answer from the given context. These contexts are fetched form corresponding Website of University of Sharjah."
            "The department in College of Science are: Department of Applied Biology offers B.Sc in Biotechnology | Department of Chemistry offers B.Sc in Chemistry | Department of Mathematics offers B.Sc in Mathematics | Department of Applied Physics and Astronomy offers B.Sc in Petroleum Geosciences and Remote Sensing and B.Sc in Applied Physics "
            
            "specific academic programs in the College of Science. "
            f"College of Science provides: {science_basicQna_Offered_Courses}"
            "For example:\n"
            "- If the student mentions 'Biotechnology' → return 'B_Sc_Bio_Technology'.\n"
            "- else if the student mentions 'Petroleum Geoscience', 'Remote Sensing', or 'Petroleum Geoscience and Remote Sensing' return 'B_Sc_Petroleum_Geoscience_and_Remote_Sensing'.\n"
            "- If the student mentions 'Applied Physics' or 'Physics' → return 'B_Sc_Applied_Physics'.\n"
            "- If the student mentions 'Chemistry' → return 'B_Sc_Chemistry'.\n"
            "- If the student mentions 'Mathematics' → return 'B_Sc_MatheMatics'.\n\n"
            "Also include a brif or a summary"
        )),
        HumanMessage(content=(
            f"The student is asking: {state.student_query}. "
            "Based on this information, determine if the student is engaging in general basic conversation about College of Science"
        ))    
    ]

    Science_basicQna_Website_Link = Science_General_QNA_linkGetter.invoke(messages)
    _Science_webLink = science_basicQna_link[Science_basicQna_Website_Link.conditionalStateGetter]

    # now fetch the content from website using the "_CCI_webLink"
    webLink_response = requests.get(_Science_webLink, timeout=60)
    webLink_response.raise_for_status()
    soup = BeautifulSoup(webLink_response.text, "html.parser")
    ScienceWEbsite_text = soup.get_text(separator=" ", strip=True)

    ScienceWEbsite_text = ScienceWEbsite_text.replace("  ", " ")
    ScienceWEbsite_text = ScienceWEbsite_text.replace("\n", " ")

    return {'Science_weblink_state': _Science_webLink, 'Science_weblink_Content': ScienceWEbsite_text}


In [1116]:
def Science_Agent_basic_qna_RAG_Processor(state: UOS_AgentState): # it will be added below CCI agent to process basic QNA using fetched text from website
    #time.sleep(40) # sleep 40 Secound
    ScienceChunked_webText = cci_science_agent_website_text_processor(state.Science_weblink_Content,  chunkSize=500)

    Science_Basic_QNA_vectorStore = FAISS.from_texts(ScienceChunked_webText, embeddings)
    Science_Basic_QNA_retriver = Science_Basic_QNA_vectorStore.as_retriever(search_kwargs={"k":5})

    Science_chain_docs = Science_Basic_QNA_retriver.get_relevant_documents(state.student_query)
    Science_chain_docs = [doc.page_content for doc in Science_chain_docs]  # list of strings
    Science_chain_docs = " ".join(Science_chain_docs)

    # now re-structure the texts 
    messages = [
        SystemMessage(content=(
            "You are an academic advisor agent for College of Science in the University of Sharjah."
            "You Use RAG system to generate answer for student and You give answer for student query by using the given fetched text from university website. "
            "Do NOT include <think> tags or reasoning traces. "
            "Only give the final structured answer directly, in human-readable form."
        )),
        HumanMessage(content=(
            f"Student Question: {state.student_query}\n"
            f"Retrieved texts: {Science_chain_docs}"
        ))
    ]

    # basic_Conversation_answer
    Science_structuredLLM_texts = generator_llm.invoke(messages)

    return {
        "Science_Basic_QNA_RAG_Answer": Science_structuredLLM_texts.content 
    }


### Medical and colleges offered courses Fetching Agent: ================

In [1117]:
def OtherCOllege_Offered_Courses_Fetch_Agent_for_CCI(state: UOS_AgentState):       
    current_offered_courses_by_AllOther_College = supabaseClient.table("courses_from_othercollege").select("*").execute().data
    remaining_courses_offered_by_other_College= []

    temp_remaining_course_in_study_plan = []
    for courses in state.Science_Course_Core_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])
    for courses in state.Science_Course_Elective_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])


    print("CCI to other college fetch!!!")
    for x in range(len(current_offered_courses_by_AllOther_College)):
        if current_offered_courses_by_AllOther_College[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_AllOther_College[x]["course_code"] in temp_remaining_course_in_study_plan:
            # filtered the un-related courses this particular student study plan 
            for key in ["id", "course_number"]:
                current_offered_courses_by_AllOther_College[x].pop(key, None) 

            remaining_courses_offered_by_other_College.append(current_offered_courses_by_AllOther_College[x])    

    return {
        "Other_College_Offered_Courses": remaining_courses_offered_by_other_College,
        "CCI_to_OtherCollge_Communication" : True
    }    
    

def OtherCOllege_Offered_Courses_Fetch_Agent_for_Science(state: UOS_AgentState): 
    current_offered_courses_by_AllOther_College = supabaseClient.table("courses_from_othercollege").select("*").execute().data
    remaining_courses_offered_by_other_College= []

    temp_remaining_course_in_study_plan = []
    for courses in state.Science_Course_Core_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])
    for courses in state.Science_Course_Elective_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])

         
    print("Science to other college fetch!!!")
    for x in range(len(current_offered_courses_by_AllOther_College)):
        if current_offered_courses_by_AllOther_College[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_AllOther_College[x]["course_code"] in temp_remaining_course_in_study_plan:
            # filtered the un-related courses this particular student study plan 
            for key in ["id", "course_number"]:
                current_offered_courses_by_AllOther_College[x].pop(key, None) 
                
            remaining_courses_offered_by_other_College.append(current_offered_courses_by_AllOther_College[x])


    return {
        "Other_College_Offered_Courses": remaining_courses_offered_by_other_College,
        "Science_to_OtherCollge_Communication" : True
    }  

           

def Medical_College_Offered_Courses_Fetch_Agent_for_CCI(state: UOS_AgentState): 
    current_offered_courses_by_Medical_College = supabaseClient.table("courses_from_medical").select("*").execute().data
    remaining_courses_offered_by_medical_College= []

    temp_remaining_course_in_study_plan = []
    for courses in state.Science_Course_Core_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])
    for courses in state.Science_Course_Elective_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])

    print("CCI to Medical college fetch!!!")
    for x in range(len(current_offered_courses_by_Medical_College)):
        if current_offered_courses_by_Medical_College[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_Medical_College[x]["course_code"] in temp_remaining_course_in_study_plan:
            # filtered the un-related courses this particular student study plan 
            for key in ["id"]:
                current_offered_courses_by_Medical_College[x].pop(key, None) 
            remaining_courses_offered_by_medical_College.append(current_offered_courses_by_Medical_College[x])  

    return {
        "Medical_College_Offered_courses": remaining_courses_offered_by_medical_College,
        "CCI_to_Medical_communication" : True
    }  




def Medical_College_Offered_Courses_Fetch_Agent_for_Science(state: UOS_AgentState): 
    current_offered_courses_by_Medical_College = supabaseClient.table("courses_from_medical").select("*").execute().data
    remaining_courses_offered_by_medical_College= []

    temp_remaining_course_in_study_plan = []
    for courses in state.Science_Course_Core_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])
    for courses in state.Science_Course_Elective_Study_Plan: 
        temp_remaining_course_in_study_plan.append(courses["course_code"])


    print("Science to Medical college fetch!!!")
    for x in range(len(current_offered_courses_by_Medical_College)):
        if current_offered_courses_by_Medical_College[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_Medical_College[x]["course_code"] in temp_remaining_course_in_study_plan:
            # filtered the un-related courses this particular student study plan 
            for key in ["id"]:
                current_offered_courses_by_Medical_College[x].pop(key, None) 

            remaining_courses_offered_by_medical_College.append(current_offered_courses_by_Medical_College[x])

    return {
        "Medical_College_Offered_courses": remaining_courses_offered_by_medical_College,
        "Science_to_Medical_communication" : True
    }  
    


     

### CCI Course scheduling Nodes Define : ==============================

In [1118]:
def CCI_Course_Scheduling_Node(state: UOS_AgentState): 
    current_offered_courses_by_CCI = supabaseClient.table("courses_from_computerscience").select("*").execute().data
    CCI_Specific_Course  = []
    CCI_Specific_Course_elective  = []

    if state.department == "BSc_Computer_Science": 
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_computer_science").select("*").execute().data
        CCI_Specific_Course_elective = supabaseClient.table("study_plan_bsc_computer_science_elective_course").select("*").execute().data

    elif state.department == "BSc_Cybersecurity": 
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_cybersecurity").select("*").execute().data

    elif state.department == "BSc_IT_Multimedia":
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_it_multimedia").select("*").execute().data
        CCI_Specific_Course_elective = supabaseClient.table("study_plan_bsc_it_multimedia_elective_course").select("*").execute().data

    elif state.department == "BSc_Biomedical_Informatics":
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_biomedical_informatics").select("*").execute().data
        CCI_Specific_Course_elective = supabaseClient.table("study_plan_bsc_biomedical_informatics_elective_course").select("*").execute().data

    elif state.department == "BSc_Computer_Engineering":
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_computer_engineering").select("*").execute().data
        CCI_Specific_Course_elective = supabaseClient.table("study_plan_bsc_computer_engineering_elective_course").select("*").execute().data

    elif state.department == "BSc_Business_Information_Systems":
        CCI_Specific_Course = supabaseClient.table("study_plan_bsc_business_information_systems").select("*").execute().data
        CCI_Specific_Course_elective = supabaseClient.table("study_plan_bsc_business_information_systems_elective_course").select("*").execute().data

        
    # now remove the courses from the study-plan that has been already taken by the student 
    remaining_course_in_study_plan = []
    remaining_elective_course_in_study_plan = []
    remaining_Study_plan_courseIds = []

    for x in range(len(CCI_Specific_Course)): 
        if CCI_Specific_Course[x]["course_code"] not in state.studentAlreadyCompletedCourses :
            for key in ["id", "course_number"]:
                    CCI_Specific_Course[x].pop(key, None) 

            remaining_course_in_study_plan.append(CCI_Specific_Course[x])
            remaining_Study_plan_courseIds.append(CCI_Specific_Course[x]["course_code"])


    if len(CCI_Specific_Course_elective) > 0: 
        # work ofr elective courses
        for x in range(len(CCI_Specific_Course_elective)): 
            if CCI_Specific_Course_elective[x]["course_code"] not in state.studentAlreadyCompletedCourses : 
                for key in ["id", "course_number"]:
                    CCI_Specific_Course_elective[x].pop(key, None) 

                remaining_elective_course_in_study_plan.append(CCI_Specific_Course_elective[x])
                remaining_Study_plan_courseIds.append(CCI_Specific_Course_elective[x]["course_code"])


    # now filter the courses offer by the College of Computing and Infromatics according to the filtered study plan
    remaining_courses_ofered_by_CCI = []
    for x in range(len(current_offered_courses_by_CCI)): 
        if current_offered_courses_by_CCI[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_CCI[x]["course_code"] in remaining_Study_plan_courseIds: 
            for key in ["id", "course_number"]:
                    current_offered_courses_by_CCI[x].pop(key, None) 
            remaining_courses_ofered_by_CCI.append(current_offered_courses_by_CCI[x])

    return {
        "CCI_Course_Core_Study_Plan" : remaining_course_in_study_plan  ,         # remaining core corses 
        "CCI_Course_Elective_Study_Plan" : CCI_Specific_Course_elective  ,       # remaining elective study plan
        "CCI_Offered_Courses_this_semester" : remaining_courses_ofered_by_CCI  , # among remaining corses srudent can select these courses 
    }
    

### CCI to Science/Medical/Other College Communication : =======================

In [1119]:
def CCI_to_Other_agent_Communication_Node(state: UOS_AgentState):
    pass

### CCI: Pass All fetched and Pre-Processed Courses to LLM to schedule: =============

In [1120]:
def CCI_Course_Scheduling_LLM_processor(state: UOS_AgentState): 
    pass

### Science Course scheduling Nodes Define : ==============================

In [1121]:
def Science_Course_Scheduling_Node(state: UOS_AgentState): 
    current_offered_courses_by_Science = supabaseClient.table("courses_from_science").select("*").execute().data
    Science_Specific_Course_elective = []
    Science_Specific_Course = []


    if state.department == "BSc_Biotechnology" : 
        Science_Specific_Course = supabaseClient.table("study_plan_bsc_biotechnology").select("*").execute().data

    elif state.department == "BSc_Petroleum_Geosciences_and_Remote_Sensing": 
        Science_Specific_Course = supabaseClient.table("study_plan_bsc_petroleum_geosciences_and_remote_sensing").select("*").execute().data
        Science_Specific_Course_elective = supabaseClient.table("study_plan_bsc_petroleum_geosciences_and_remote_sensing_electiv").select("*").execute().data

    elif state.department == "BSc_Applied_Physics" :
        Science_Specific_Course = supabaseClient.table("study_plan_bsc_applied_physics").select("*").execute().data
        Science_Specific_Course_elective = supabaseClient.table("study_plan_bsc_applied_physics_electives").select("*").execute().data
        
    elif state.department == "BSc_Chemistry" :
        Science_Specific_Course = supabaseClient.table("study_plan_bsc_chemistry").select("*").execute().data

    elif state.department == "BSc_Mathematics" :
        Science_Specific_Course = supabaseClient.table("study_plan_bsc_mathematics").select("*").execute().data
        Science_Specific_Course_elective = supabaseClient.table("study_plan_bsc_mathematics_elective_courses").select("*").execute().data
        print("Math sourse: ", len(Science_Specific_Course) , len(Science_Specific_Course_elective))

    
    # now remove the courses from the study-plan that has been already taken by the student 
    remaining_course_in_study_plan = []
    remaining_elective_course_in_study_plan = []
    remaining_Study_plan_courseIds = []


    # work for the core study study plan
    for x in range(len(Science_Specific_Course)): 
        if Science_Specific_Course[x]["course_code"] not in state.studentAlreadyCompletedCourses :
            for key in ["id", "course_number"]:
                    Science_Specific_Course[x].pop(key, None) 

            remaining_course_in_study_plan.append(Science_Specific_Course[x])
            remaining_Study_plan_courseIds.append(Science_Specific_Course[x]["course_code"])


    if len(Science_Specific_Course_elective) > 0: 
        # work ofr elective courses
        for x in range(len(Science_Specific_Course_elective)): 
            if Science_Specific_Course_elective[x]["course_code"] not in state.studentAlreadyCompletedCourses : 
                for key in ["id", "course_number"]:
                    Science_Specific_Course_elective[x].pop(key, None) 

                remaining_elective_course_in_study_plan.append(Science_Specific_Course_elective[x])
                remaining_Study_plan_courseIds.append(Science_Specific_Course_elective[x]["course_code"])


    # now filter the courses offer by the College of science according to the filtered study plan
    remaining_courses_ofered_by_Science = []
    for x in range(len(current_offered_courses_by_Science)): 
        if current_offered_courses_by_Science[x]["course_code"] not in state.studentAlreadyCompletedCourses and current_offered_courses_by_Science[x]["course_code"] in remaining_Study_plan_courseIds: 
            for key in ["id", "course_number"]:
                    current_offered_courses_by_Science[x].pop(key, None) 
            remaining_courses_ofered_by_Science.append(current_offered_courses_by_Science[x])

    return {
        "Science_Course_Core_Study_Plan" : remaining_course_in_study_plan  , 
        "Science_Course_Elective_Study_Plan" : Science_Specific_Course_elective  , 
        "Science_Offered_Courses_this_semester" : remaining_courses_ofered_by_Science  
    }

### Science to CCI/Medical/Other College Communication : =======================

In [1122]:
def Science_to_Other_agent_Communication_Node(state: UOS_AgentState):         
    pass


### Science: Pass All fetched and Pre-Processed Courses to LLM to schedule: =============

In [1123]:
def Science_Course_Scheduling_LLM_processor(state: UOS_AgentState): 
    pass

### add conditional statement for splitting nodes: ======================

In [1124]:
# routing model for Top level agent 
def top_level_agent_routting_model(state: UOS_AgentState): 
    if state.topLevelAgent_basicConversationOrNot == False and state.UOS_AgentState_top_level_decision == "CCIAgent": 
        return "CCIAgent_Route"
    elif state.topLevelAgent_basicConversationOrNot == False and state.UOS_AgentState_top_level_decision == "ScienceAgent":
        return "ScienceAgent_Route"
    else: 
        return "topLevel_agent_rag_workflow"


# routing model for CCI agent 
def CCI_agent_routting_model(state: UOS_AgentState):
    if state.UOS_AgentState_CCIAgent_decision == "CCI_Conversation": 
        return "CCI_Agent_Basic_QNA_Route"
    elif state.UOS_AgentState_CCIAgent_decision == "CCI_CourseScheduling_advising": 
        return "CCI_Agent_CourseScheduling_Route"


# routing model for Science agent 
def Science_agent_routing_model(state: UOS_AgentState):
    if state.UOS_AgentState_ScienceAgent_decision == "Science_Conversation": 
        return "Science_Agent_Basic_QNA_Route"
    elif state.UOS_AgentState_ScienceAgent_decision == "Science_CourseScheduling_advising": 
        return "Science_Agent_CourseScheduling_Route"
        



# CCI agent to Science/Medical/otherCollege iterative Communication
def CCI_agent_communication_routing_model(state: UOS_AgentState): 
    
    if state.CCI_to_Medical_communication == False: 
        return "CCI_to_Medical_Communication_trigger"
    
    elif state.CCI_to_OtherCollge_Communication == False: 
        return "CCI_to_OtherCollege_communication_trigger"
    
    elif state.CCI_to_Medical_communication == True and state.CCI_to_OtherCollge_Communication==True: 
        return "Send_all_Fetched_Courses_in_CCI_LLM"
    


# Science agent to CCI/Medical/otherCollege iterative Communication
def Science_agent_communication_routing_model(state: UOS_AgentState): 
    if state.Science_to_Medical_communication == False: 
        return "Science_to_Medical_Communication_trigger"
    
    elif state.Science_to_OtherCollge_Communication == False: 
        return "Science_to_OtherCollege_communication_trigger"
    
    elif state.Science_to_Medical_communication == True and state.Science_to_OtherCollge_Communication == True : 
        return "Send_all_Fetched_Courses_in_Science_LLM"



# Medical College Node routing Node
def MedicalCollege_routing_model(state: UOS_AgentState): 
    if state.CCI_to_Medical_communication == True: 
        return "fetched_Medical_Courses_for_CCI"
    elif state.Science_to_Medical_communication == True: 
        return "fetched_Medical_Courses_for_Science"
    


# Other all Colleges Node Routing Node
def OtherCollege_Routing_model(state: UOS_AgentState): 
    if state.CCI_to_OtherCollge_Communication == True: 
        return "fetched_OtherCollege_Courses_for_CCI"
    elif state.Science_to_OtherCollge_Communication == True: 
        return "fetched_OtherCollege_Courses_for_Science"


    

### define the Nodes and Edges and Connect them

In [1125]:
graph = StateGraph(UOS_AgentState)


graph.add_node('uosTopAgent', topLevelLLM_call)
graph.add_node('uosTopAgent_Basic_QNA_RAG', topLevel_agent_RAG)

# CCI Nodes 
graph.add_node('uosCCIAgent', CCI_LLM_call)
graph.add_node('CCI_Agent_basic_qna_Node', CCI_Agent_basic_qna_Node_Fn)
graph.add_node('CCI_Agent_basic_qna_RAG_Node', CCI_Agent_basic_qna_RAG_Processor)
graph.add_node('CCI_Agent_Course_Scheduling', CCI_Course_Scheduling_Node)
graph.add_node('CCI_to_Other_agent_Communication_Node', CCI_to_Other_agent_Communication_Node) # CCI to all college communication Node
graph.add_node('CCI_Course_Scheduling_LLM_processor', CCI_Course_Scheduling_LLM_processor)

# Science Nodes
graph.add_node('uosScienceAgent', Science_LLM_call)
graph.add_node('Science_Agent_basic_qna_Node', Science_Agent_basic_qna_Node_Fn)
graph.add_node('Science_Agent_basic_qna_Rag_Node', Science_Agent_basic_qna_RAG_Processor)
graph.add_node('Science_Agent_Course_Scheduling', Science_Course_Scheduling_Node)
graph.add_node('Science_to_Other_agent_Communication_Node', Science_to_Other_agent_Communication_Node)  # science to all college communication Node
graph.add_node('Science_Course_Scheduling_LLM_processor', Science_Course_Scheduling_LLM_processor)

# Other college and Medical College Nodes
graph.add_node('OtherCOllege_Offered_Courses_Fetch_Agent_for_CCI', OtherCOllege_Offered_Courses_Fetch_Agent_for_CCI)
graph.add_node('OtherCOllege_Offered_Courses_Fetch_Agent_for_Science', OtherCOllege_Offered_Courses_Fetch_Agent_for_Science)
graph.add_node('Medical_College_Offered_Courses_Fetch_Agent_For_CCI', Medical_College_Offered_Courses_Fetch_Agent_for_CCI)
graph.add_node('Medical_College_Offered_Courses_Fetch_Agent_For_Science', Medical_College_Offered_Courses_Fetch_Agent_for_Science)

# define edges
graph.add_edge(START, 'uosTopAgent')
graph.add_conditional_edges(
    'uosTopAgent', 
    top_level_agent_routting_model , 
    {
        'CCIAgent_Route': 'uosCCIAgent', 
        'ScienceAgent_Route': 'uosScienceAgent', 
        'topLevel_agent_rag_workflow': 'uosTopAgent_Basic_QNA_RAG'
    }
)
graph.add_conditional_edges(
    'uosCCIAgent', 
    CCI_agent_routting_model, 
    {
        'CCI_Agent_Basic_QNA_Route': 'CCI_Agent_basic_qna_Node' , 
        'CCI_Agent_CourseScheduling_Route': 'CCI_Agent_Course_Scheduling'
    }
)
graph.add_conditional_edges(
    'uosScienceAgent', 
    Science_agent_routing_model, 
    {
        'Science_Agent_Basic_QNA_Route': 'Science_Agent_basic_qna_Node' , 
        'Science_Agent_CourseScheduling_Route': 'Science_Agent_Course_Scheduling'
    }
)

graph.add_edge('uosTopAgent_Basic_QNA_RAG', END)



# CCI end Nodes : =====================================================================
graph.add_edge('CCI_Agent_basic_qna_Node', 'CCI_Agent_basic_qna_RAG_Node')
graph.add_edge('CCI_Agent_basic_qna_RAG_Node', END)
graph.add_edge('CCI_Agent_Course_Scheduling', 'CCI_to_Other_agent_Communication_Node')

#####  CCI_to_Science_Communication: bool = False 
#####  CCI_to_Medical_communication: bool = False 
#####  CCI_to_OtherCollge_Communication: bool = False

graph.add_conditional_edges(
    'CCI_to_Other_agent_Communication_Node', 
    CCI_agent_communication_routing_model, 
    {
        'CCI_to_Medical_Communication_trigger' : 'Medical_College_Offered_Courses_Fetch_Agent_For_CCI',
        'CCI_to_OtherCollege_communication_trigger' : 'OtherCOllege_Offered_Courses_Fetch_Agent_for_CCI', 
        'Send_all_Fetched_Courses_in_CCI_LLM' : 'CCI_Course_Scheduling_LLM_processor'
    }
)
graph.add_edge('Medical_College_Offered_Courses_Fetch_Agent_For_CCI', 'CCI_to_Other_agent_Communication_Node')
graph.add_edge('OtherCOllege_Offered_Courses_Fetch_Agent_for_CCI', 'CCI_to_Other_agent_Communication_Node')
graph.add_edge('CCI_Course_Scheduling_LLM_processor', END)





# science end nodes: ==================================================================
graph.add_edge('Science_Agent_basic_qna_Node', 'Science_Agent_basic_qna_Rag_Node')
graph.add_edge('Science_Agent_basic_qna_Rag_Node', END)
graph.add_edge('Science_Agent_Course_Scheduling', 'Science_to_Other_agent_Communication_Node')

###### Science_to_CCI_Communication: bool = False 
###### Science_to_Medical_communication: bool = False 
###### Science_to_OtherCollge_Communication: bool = False

graph.add_conditional_edges(
    'Science_to_Other_agent_Communication_Node', 
    Science_agent_communication_routing_model, 
    {
        'Science_to_Medical_Communication_trigger' : 'Medical_College_Offered_Courses_Fetch_Agent_For_Science',
        'Science_to_OtherCollege_communication_trigger' : 'OtherCOllege_Offered_Courses_Fetch_Agent_for_Science', 
        'Send_all_Fetched_Courses_in_Science_LLM' : 'Science_Course_Scheduling_LLM_processor'
    }
)

graph.add_edge('Medical_College_Offered_Courses_Fetch_Agent_For_Science', 'Science_to_Other_agent_Communication_Node')
graph.add_edge('OtherCOllege_Offered_Courses_Fetch_Agent_for_Science', 'Science_to_Other_agent_Communication_Node')
graph.add_edge('Science_Course_Scheduling_LLM_processor', END)




# compile the graph
mainWorkingGaph = graph.compile()

In [1131]:
from IPython.display import display, HTML
import uuid

div_id = f"mermaid-{uuid.uuid4().hex}"

# Get Mermaid code from your graph
mermaid_code = mainWorkingGaph.get_graph().draw_mermaid()

html_code = f"""
<div style="width:3000px; height:auto;">
  <div id="{div_id}" class="mermaid">
    {mermaid_code}
  </div>
</div>

<script type="module">
import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
mermaid.initialize({{ startOnLoad:true }});
mermaid.run({{ 
    nodes: document.querySelectorAll('#{div_id}')
}});
</script>
"""

display(HTML(html_code))



### Now do some dummy call

        "BSc_Mathematics", 
        "BSc_Chemistry", 
        "BSc_Petroleum_Geosciences_and_Remote_Sensing", 
        "BSc_Biotechnology", 
        "BSc_Business_Information_Systems", 
        "BSc_IT_Multimedia", 
        "BSc_Computer_Science", 
        "BSc_Cybersecurity", 
        "BSc_Computer_Engineering",
        "BSc_Applied_Physics"

In [1128]:
initial_state_CCI = {
    "uid": "U22106802", 
    "mensOrWomCampus" : "mens",
    "standing": "Senior",
    "semester": 7,
    "yearOfStudy": 4,
    "college": "College of Computing and Informatics", 
    "department" : "BSc_Computer_Science",
    "studentAlreadyCompletedCourses": [
        "1420-101", "1420-102", "1430-110",	"1430-116",
        "1440-131","1501-116","1501-211",
        "1501-215","1501-246","1501-250","1501-252",
        "1501-279","1501-371","1501-394","1501-454",
        "1501-341","1501-465","1440-211","1440-281",
        "0104-100","0302-200","1501-100","0302-150","0401-142"
    ],
    "StudentAlreadyCompletedCredits" : 72,
    #"student_query": "How many credits do I need to complete to graduate if I am majoring in Bio-technology?",
    #"student_query": "How many credits do I need to graduate if I am majoring in Computer Science?",
    #"student_query": "Can you register 5 courses offerd by university this semester for me? ",
    #"student_query": "How many colleges are there in university of sharjah?",
    #"student_query": "is UAE Society compulsory course in Computer Science course? ",
    #"student_query": "What are the courses needs to take in 6th semester if I am majoring in Computer Science?",
    "student_query": "Can you please schedule the courses for this semester for me? ",
}


initial_state_Science = {
    "uid": "U22106802", 
    "mensOrWomCampus" : "mens",
    "standing": "Sophomore",
    "semester": 3,
    "yearOfStudy": 4,
    "college": "College of Science", 
    "department" : "BSc_Biotechnology",
    "studentAlreadyCompletedCourses": [
        "0201-102", "0202-112", "1420-101",	"1420-102",
        "1430-110","1430-116","1440-131",
        "1501-100","1420-103",
        "1420-104","1430-117",
        "1430-118","1440-132","1420-221",
        "1420-222","0204-102",
        "0104-101","1420-361",
    ],
    "StudentAlreadyCompletedCredits" : 36,
    #"student_query": "How many credits do I need to complete to graduate if I am majoring in Bio-technology?",
    #"student_query": "How many credits do I need to graduate if I am majoring in Computer Science?",
    #"student_query": "Can you register 5 courses offerd by university this semester for me? ",
    #"student_query": "How many colleges are there in university of sharjah?",
    #"student_query": "is UAE Society compulsory course in Computer Science course? ",
    #"student_query": "What are the courses needs to take in 6th semester if I am majoring in Computer Science?",
    "student_query": "Can you please schedule the courses for this semester for me? ",
}



result = mainWorkingGaph.invoke(initial_state_Science)
display(result)

UOS_AgentState_ScienceAgent_decision:  Science_CourseScheduling_advising
Science to Medical college fetch!!!
Science to other college fetch!!!


{'uid': 'U22106802',
 'mensOrWomCampus': 'mens',
 'standing': 'Sophomore',
 'yearOfStudy': 4,
 'semester': 3,
 'college': 'College of Science',
 'department': 'BSc_Biotechnology',
 'studentAlreadyCompletedCourses': ['0201-102',
  '0202-112',
  '1420-101',
  '1420-102',
  '1430-110',
  '1430-116',
  '1440-131',
  '1501-100',
  '1420-103',
  '1420-104',
  '1430-117',
  '1430-118',
  '1440-132',
  '1420-221',
  '1420-222',
  '0204-102',
  '0104-101',
  '1420-361'],
 'StudentAlreadyCompletedCredits': 36,
 'student_query': 'Can you please schedule the courses for this semester for me? ',
 'UOS_AgentState_top_level_decision': 'ScienceAgent',
 'topLevelAgent_basicConversationOrNot': False,
 'UOS_AgentState_ScienceAgent_decision': 'Science_CourseScheduling_advising',
 'Science_Course_Core_Study_Plan': [{'course_code': '0104-100',
   'course_title': 'Islamic Culture (1)',
   'prerequisites': [],
   'course_semester': 'Year 1 - 1st semester'},
  {'course_code': '1426-155',
   'course_title': 'Ge

In [1129]:
try: 
    print(len(str(result["CCI_Course_Core_Study_Plan"]).split(" ")))
    print(len(str(result["CCI_Course_Elective_Study_Plan"]).split(" ")))
    print(len(str(result["CCI_Offered_Courses_this_semester"]).split(" ")))
    print(len(str(result["Medical_College_Offered_courses"]).split(" ")))
    print(len(str(result["Other_College_Offered_Courses"]).split(" ")))
    print("CCI course: ")
except : 
    print(len(str(result["Science_Course_Core_Study_Plan"]).split(" ")))
    print(len(str(result["Science_Course_Elective_Study_Plan"]).split(" ")))
    print(len(str(result["Science_Offered_Courses_this_semester"]).split(" ")))
    print(len(str(result["Medical_College_Offered_courses"]).split(" ")))
    print(len(str(result["Other_College_Offered_Courses"]).split(" ")))
    print("Science course: ")

597
1
1148
382
1541
Science course: 
